In [2]:
import numpy as np
import SimpleITK as sitk
from typing import Tuple
import skimage as sk
from PIL import Image
import napari
import matplotlib.pyplot as plt
import scipy as sc
from glob import glob
#from Get_Orientation import get_cardinal_points, detect_cardinal_point, compute_transform
import pandas as pd
import os

In [94]:
def get_tissue_bbox(img,name,tissue_id,data_path):
    mask = img < 255
    labels = sk.measure.label(mask)
    props = sk.measure.regionprops(labels)
    largest_obj = max(props, key=lambda p: p.area)
    largest_obj_label = largest_obj.label
    minc = min(props, key=lambda p: p.bbox[0])
    minr = min(props, key=lambda p: p.bbox[1])
    maxc = max(props, key=lambda p: p.bbox[2])
    maxr = max(props, key=lambda p: p.bbox[3])
    img_bbox = img[minc.bbox[0]-10:maxc.bbox[2]+10,minr.bbox[1]-10:maxr.bbox[3]+10]
    labels_bbox = labels[minc.bbox[0]-10:maxc.bbox[2]+10,minr.bbox[1]-10:maxr.bbox[3]+10]
    cropped_mask = (labels_bbox==largest_obj_label)*labels_bbox
    save_path_img = os.path.join(data_path,tissue_id,'cropped_image')
    save_path_mask = os.path.join(data_path,tissue_id,'cropped_mask')
    os.makedirs(save_path_img,exist_ok=True)
    os.makedirs(save_path_mask,exist_ok=True)
    sk.io.imsave(os.path.join(save_path_img,'cropped_'+str(name)+'.png'),img_bbox,check_contrast=False)
    sk.io.imsave(os.path.join(save_path_mask,'cropped_mask_'+str(name)+'.png'),cropped_mask,check_contrast=False)
    return img_bbox, cropped_mask,save_path_img,save_path_mask



In [95]:
def add_padding(img_bbox,cropped_mask,name,data_path):
    """ 
    reshape the cropped images to a square based on the largest dim using padding
    saves new arrays with added padding of a constant value: 255 for grey scale image, 0 for masks
    returns save path for later use
    """
    max_dim = max(img_bbox.shape)
    padding_y = (max_dim - img_bbox.shape[0]) // 2
    padding_x = (max_dim - img_bbox.shape[1]) // 2
    pad_img_bbox = np.pad(img_bbox,((padding_y+50,padding_y+50),(padding_x+50,padding_x+50)),mode='constant',constant_values=255)
    pad_mask_bbox = np.pad(cropped_mask,((padding_y+50,padding_y+50),(padding_x+50,padding_x+50)),mode='constant',constant_values=0)
    save_path_img = os.path.join(data_path,tissue_id,'padded_cropped_image')
    save_path_mask = os.path.join(data_path,tissue_id,'padded_cropped_mask')
    os.makedirs(save_path_img,exist_ok=True)
    os.makedirs(save_path_mask,exist_ok=True)
    sk.io.imsave(os.path.join(save_path_img,'padded_cropped_'+str(name)+'.png'),pad_img_bbox,check_contrast=False)
    sk.io.imsave(os.path.join(save_path_mask,'padded_cropped_mask_'+str(name)+'.png'),pad_mask_bbox,check_contrast=False)
    return pad_img_bbox,pad_mask_bbox,save_path_img,save_path_mask



In [107]:
def get_map_region(data_path,map_base,map_id,grey_value_df):
    path = os.path.join(data_path,'Maps',map_base+'.png')
    map_arr = np.array(Image.open(path).convert('L'))
    grey_value = grey_value_df.loc[grey_value_df['Mapping_ID'] == map_id, 'Grey_value'].values[0] 
    map_region = (map_arr==grey_value)*map_arr
    return map_region

In [6]:
def detect_cardinal_point(arr, grey_value):
    """
    Detect the centroid of a cardinal point marker by its grey value.
    
    Returns (x, y) in pixel coordinates, or None if not found.
    """
    
    # Create binary mask for this grey value
    mask = (arr * (arr == grey_value)).astype(int)
        
    if not mask.any():
        print(f"  WARNING: No pixels found for grey value {grey_value} in {name}")
        return None
    
    # Label connected components and get centroid
    labeled_img = sk.measure.label(mask)
    props = sk.measure.regionprops(labeled_img)
    cy,cx = props[0].centroid
    return np.array([int(cy), int(cx)])  # return as (y,x)


In [108]:
def get_cardinal_points(img, name,grey_value_df):
    """
    Extract both cardinal points from an image.
    Returns dict with 'north' and 'east' as (y,x) arrays.
    """
    points = {}
    for direction in ["north", "east"]:
        grey = grey_value_df.loc[grey_value_df['Mapping_ID'] == direction, 'Grey_value'].values[0]
        pt = detect_cardinal_point(img, grey)
        if pt is None:
            raise ValueError(f"Could not find '{direction}' point in {name}")
        points[direction] = pt
        print(f"  {direction}: pixel (y,x) ({pt[0]:.1f}, {pt[1]:.1f})")
    return points


In [8]:
def orient_tissue(mask_points,moving_image):
    """
    Determine the flip and/or rotation needed to orient the moving image
    to match the map, using the N and E cardinal point locations.

    Detection logic (image coordinates, y increases downward):
        North marker should be visually "above" East  → north_y < east_y
        East  marker should be visually "right" of North → east_x > north_x

    Flip cases:
        north_is_up  and east_is_right  → none
        north_is_up  and east_is_left → horizontal flip
        north_is_right and east_is_up  → -90 rotation
        north_is_right and east_is_down → 90 rotation
        north_is_left and east_is_up → Vertical flip + 90 rotation
        north_is_left and east_is_down → Vertical flip + -90 rotation
        north_is_down and east_is_left → vertical + horizontal flip
        north_is_down and east_is_right → vertical flip

    Parameters
    ----------
    mask_points  : dict with 'north' and 'east' as (x, y) pixel arrays — moving image
    moving_image: image array of moving image
    Returns
    -------
    flip matrix
    """
    # --- Detect required flip from cardinal point geometry ---
    # Compare pixel coords directly for orientation — spacing does not
    # affect which direction is "up" or "right"
    #get moving image relative cardinal coordinates:
    north = (0, moving_image.shape[1]//2)
    south = (moving_image.shape[0]-1, moving_image.shape[1]//2)
    east = (moving_image.shape[0]//2, moving_image.shape[1]-1)
    west = (moving_image.shape[0]//2, 0)
    cardinals = {"north": north, "south": south, "east": east, "west": west}

    #which cardinal direction are the two reference points closest to?
    closest_north = min(cardinals, 
                  key=lambda d: np.hypot(mask_points['north'][0] - cardinals[d][0], 
                    mask_points['north'][1] - cardinals[d][1]))
    closest_east = min(cardinals, 
                  key=lambda d: np.hypot(mask_points['east'][0] - cardinals[d][0], 
                    mask_points['east'][1] - cardinals[d][1]))

    if closest_north == 'north' and closest_east == 'east':
        flip = None
        rotation = None
    elif closest_north == 'north' and closest_east == 'west':
        flip = 'horizontal'
        rotation = None
    elif closest_north == 'west' and closest_east == 'north':
        flip = None
        rotation = 'clockwise'
    elif closest_north == 'east' and closest_east == 'north':
        flip = 'vertical'
        rotation = 'counter-clockwise'
    elif closest_north == 'west' and closest_east == 'south':
        flip = 'vertical'
        rotation = 'clockwise'
    elif closest_north == 'east' and closest_east == 'south':
        flip = None
        rotation = 'counter-clockwise'
    elif closest_north == 'south' and closest_east == 'west':
        flip = 'both'
        rotation = None
    elif closest_north == 'south' and closest_east == 'east':
        flip = 'vertical'
        rotation = None
    else:
        print('Orientation does not match conditions, check image')
    print(f'Detected Flip: {flip}')
    print(f'Detected Rotation: {rotation}')
    return flip, rotation

In [ ]:
def apply_flip_rotation(sitk_moving,sitk_fixed,flip=None,rotation=None):
    """ 
    Align moving image to map
    """
    FLIP_MATRICES = {
    "none":       [ 1,  0, 0,  1],
    "horizontal": [-1,  0, 0,  1],
    "vertical":   [1,  0, 0, -1],
    "both":       [-1,  0, 0, -1],
    }
    ROTATION_DICTIONARY = {
        'clockwise' : -90,
        'counter-clockwise': 90
    }
    if flip != None and rotation != None:
        print("Applying flip and rotation transform")
        moving_center = sitk_moving.GetOrigin() + (np.array(sitk_moving.GetSpacing()) * np.array(sitk_moving.GetSize()) / 2.0)
        #apply flip with Affine
        flip_transform = sitk.AffineTransform(sitk_moving.GetDimension())
        flip_transform.SetMatrix(FLIP_MATRICES[flip])
        flip_transform.SetCenter(moving_center)
        #apply rotation with Euler
        angle = ROTATION_DICTIONARY[rotation]
        rads = np.deg2rad(angle)
        rotation_transform = sitk.Euler2DTransform()
        rotation_transform.SetCenter(moving_center)
        rotation_transform.SetAngle(rads)
        initial_transform = sitk.CenteredTransformInitializer(sitk_fixed,sitk_moving,
                    sitk.AffineTransform(sitk_moving.GetDimension()),
                    sitk.CenteredTransformInitializerFilter.MOMENTS)
        composite_transform = sitk.CompositeTransform(2)
        composite_transform.AddTransform(initial_transform)
        composite_transform.AddTransform(rotation_transform)
        composite_transform.AddTransform(flip_transform)

    elif flip == None and rotation != None:
        print("Applying rotation transform")
        moving_center = sitk_moving.GetOrigin() + (np.array(sitk_moving.GetSpacing()) * np.array(sitk_moving.GetSize()) / 2.0)
        #apply rotation with Euler
        angle = ROTATION_DICTIONARY[rotation]
        rads = np.deg2rad(angle)
        rotation_transform = sitk.Euler2DTransform()
        rotation_transform.SetCenter(moving_center)
        rotation_transform.SetAngle(rads)
        initial_transform = sitk.CenteredTransformInitializer(sitk_fixed,sitk_moving,
                    sitk.AffineTransform(sitk_moving.GetDimension()),
                    sitk.CenteredTransformInitializerFilter.MOMENTS)
        composite_transform = sitk.CompositeTransform(2)
        composite_transform.AddTransform(initial_transform)
        composite_transform.AddTransform(rotation_transform)
    
    elif flip != None and rotation == None:
        print("Applying flip transform")
        moving_center = sitk_moving.GetOrigin() + (np.array(sitk_moving.GetSpacing()) * np.array(sitk_moving.GetSize()) / 2.0)
        #apply flip with Affine
        flip_transform = sitk.AffineTransform(sitk_moving.GetDimension())
        flip_transform.SetMatrix(FLIP_MATRICES[flip])
        flip_transform.SetCenter(moving_center)
        initial_transform = sitk.CenteredTransformInitializer(sitk_fixed,sitk_moving,
                    sitk.AffineTransform(sitk_moving.GetDimension()),
                    sitk.CenteredTransformInitializerFilter.MOMENTS)
        composite_transform = sitk.CompositeTransform(2)
        composite_transform.AddTransform(initial_transform)
        composite_transform.AddTransform(flip_transform)
    else:
        print("Tissue in correct orientation, no flip or rotation applied")
        initial_transform = sitk.CenteredTransformInitializer(sitk_fixed,sitk_moving,
                    sitk.AffineTransform(sitk_moving.GetDimension()),
                    sitk.CenteredTransformInitializerFilter.MOMENTS)
        composite_transform = sitk.CompositeTransform(2)
        composite_transform.AddTransform(initial_transform)
    return composite_transform

def multi_start_registration(sitk_moving, sitk_fixed,data_path,tissue_id,composite_transform):
    #Finetune alignment with test angles
    save_path = os.path.join(data_path,tissue_id,'transformation_files')
    save_file = os.path.join(save_path,name+'_'+tissue_id+'.tfm')
    os.makedirs(save_path,exist_ok=True)
    angles = [-20,-15,-10,-5,0,5,10,15,20]
    rads = [np.deg2rad(a) for a in angles]
    best_metric    = np.inf
    best_transform = None
    best_angle = 0
    for i, r in enumerate(rads):
        moving_center = sitk_moving.GetOrigin() + (np.array(sitk_moving.GetSpacing()) * np.array(sitk_moving.GetSize()) / 2.0)
        test_rotation_transform = sitk.Euler2DTransform()
        test_rotation_transform.SetCenter(moving_center)
        test_rotation_transform.SetAngle(r)
        
        test_composite_transform = sitk.CompositeTransform(2)
        test_composite_transform.AddTransform(test_rotation_transform)
        test_composite_transform.AddTransform(composite_transform)
        registration_method = sitk.ImageRegistrationMethod()
        registration_method.SetMetricAsMeanSquares()
        registration_method.SetInitialTransform(test_composite_transform,inPlace=False)
        registration_method.SetOptimizerAsLBFGS2(
            solutionAccuracy=1e-5,
            numberOfIterations=500,
            deltaConvergenceTolerance=0.01
            )
        registration_method.SetOptimizerScalesFromPhysicalShift()
        registration_method.SetShrinkFactorsPerLevel(shrinkFactors=[4, 2, 1])
        registration_method.SetSmoothingSigmasPerLevel(smoothingSigmas=[2, 1, 0])
        registration_method.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
        registration_method.SetInterpolator(sitk.sitkNearestNeighbor)
        #registration_method.AddCommand(sitk.sitkIterationEvent, lambda: command_iteration(registration_method))
        print(f"  Registering...")
        print(f"  Start {i+1}/{len(rads)} (angle={np.degrees(r):.0f}°)")
        result = registration_method.Execute(sitk_fixed, sitk_moving)
        metric = registration_method.GetMetricValue()
        print(f"metric: {metric:.6f}")
                    
        if metric < best_metric:
            best_metric    = metric
            best_transform = result
            best_angle = angles[i]

        #optimized_transform = registration_method.Execute(sitk_fixed,sitk_moving)
    sitk.WriteTransform(best_transform,save_file)
    print(f"Angle: {best_angle} gave the best metric: {best_metric:.6f}")
    return best_transform

In [10]:
def calculate_spacing(img_mask,map_region_mask):
    #get area of tissue mask bounding box
    moving_labels = sk.measure.label(img_mask)
    props = sk.measure.regionprops(moving_labels)
    minc,minr,maxc,maxr = props[0].bbox
    moving_height_px = maxr - minr
    moving_width_px  = maxc - minc
    moving_area_px = moving_height_px * moving_width_px
    #get area of map region bounding box
    map_region_label = sk.measure.label(map_region_mask)
    map_region_props = sk.measure.regionprops(map_region_label)
    minc,minr,maxc,maxr = map_region_props[0].bbox
    map_height_px = maxr - minr
    map_width_px  = maxc - minc
    map_area_px = map_height_px * map_width_px
    spacing = round(np.sqrt(moving_area_px//map_area_px),1)
    map_spacing = (float(spacing),float(spacing))
    return map_spacing

In [ ]:
def load_sitk_moving_img(moving_img_arr,moving_spacing):
    """ Convert array to 32bit float and then to sitk image for registration """
    bitdepth = np.array(moving_img_arr).astype(np.float32)
    sitk_img = sitk.GetImageFromArray(bitdepth)
    sitk_img.SetSpacing(moving_spacing)
    return sitk_img

def load_sitk_fixed_img(fixed_img_arr,fixed_spacing):
    """ Convert array to 32bit float and then to sitk image for registration """
    bitdepth = np.array(fixed_img_arr).astype(np.float32)
    sitk_img = sitk.GetImageFromArray(bitdepth)
    sitk_img.SetSpacing(fixed_spacing)
    return sitk_img
    

Mammary gland sizes:
Small gland:
width: 4.1cm
height: 5.3 cm
Large gland:
Width: 4.3 cm
Height: 6.1 cm

In [ ]:
#get mapID grey values
maps = sorted(glob('Maps\PNGsForReg\GrayscaleMaps\*.png'))
names = list(map(os.path.basename,maps))
map_arrays = [np.array(Image.open(map).convert('L')) for map in maps]



In [58]:
avg_y = round(float(np.average([5.3,6.1])),2)
avg_x = round(float(np.average([4.1,4.3])),2)

In [72]:
round(0.5031*32,2)

16.1

In [67]:
#get spacing for maps:
map_mask = map_arrays[0] < 255
map_labels = sk.measure.label(map_mask)
props = sk.measure.regionprops(map_labels)
map_region = max(props, key=lambda p: p.area)
minc,minr,maxc,maxr = map_region.bbox
map_height_px = maxr - minr
map_width_px  = maxc - minc
y_spacing_um = round((map_height_px/avg_y)*10000,2)
x_spacing_um = round((map_width_px/avg_x)*10000,2)

In [73]:
pixel_size = {
    'map spacing' : {'yspacing': y_spacing_um,
    'xspacing': x_spacing_um},
    'image spacing' : {'yspacing': 16.1,
    'xspacing': 16.1}
}

In [74]:
import json
file = 'pixel_spacing.json'
with open(file, 'w') as json_file:
    json.dump(pixel_size, json_file, indent=4)

In [50]:
for i, name in enumerate(names):
    print(f'{i}. {name}')

0. 3 _missinglowA.png
1. 3 _missinglowB.png
2. 3 _missinglowC.png
3. 3 _missinglowD.png
4. 3 _missinglowE.png
5. 4_Swapped2A.png
6. 4_Swapped2B.png
7. 5_Swapped2A.png
8. 5_Swapped2B.png
9. 6139_Left.png
10. 6544_Left.png
11. 6545_Right.png
12. 6775_Left.png
13. 6839_Right.png
14. 7311_Right.png
15. 7373_Right.png
16. 7529_Right.png
17. 7603_Right.png
18. 7778_Left.png
19. 7841_Left.png
20. 7844_Left.png
21. 7948_Right.png
22. 8125_Left.png
23. 8132_Right.png
24. 8183_Right.png
25. 8434_Left.png
26. Average2.png
27. Average3.png
28. Average4.png
29. Average5.png
30. Average6.png
31. Nulliparous.png
32. SD183_Left.png
33. SD183_Right.png
34. SD187_Right.png
35. SD194_Left.png
36. SD211_Left.png
37. SD215_Left.png
38. SD274_Right.png
39. SD351_Right.png
40. SD401_Left.png
41. SD7212_Right.png
42. SD7215.png
43. Stacked.png


In [32]:
largest_array = max(map_arrays, key=lambda p: p.shape[0]*p.shape[1])
smallest_array = min(map_arrays, key=lambda p: p.shape[0]*p.shape[1])
print(largest_array.shape, smallest_array.shape)

(1054, 914) (1052, 912)


In [46]:
matched_size = [sk.transform.resize(map,(1100,1000),mode='constant',cval=255,preserve_range=True) for map in map_arrays]

In [47]:
stacked_maps = np.stack(matched_size)
viewer = napari.view_image(stacked_maps)

E:\Temp\ipykernel_40204\3376961061.py:2: FutureWarning: `napari.view_image` is deprecated and will be removed in napari 0.7.0.
Use `viewer = napari.Viewer(); viewer.add_image(...)` instead.
  viewer = napari.view_image(stacked_maps)


In [101]:
#set up save locations:
data_path = 'Pipeline_Test'
df_key = pd.read_csv('Annotation_Key.csv',dtype=str,usecols=['Image','Genotype','Tissue.ID','MappingID','MapBase','AnimalID'])
grey_value_df = pd.read_csv('Maps\PNGsForReg\HexColorMappingKey.2026.2.28.csv',dtype=str,usecols=['Mapping_ID','Grey_value'])

In [102]:
df_key.head()

,Image,Genotype,AnimalID,Tissue.ID,MappingID,MapBase
0,156963,WT,1103,Tissue,1,Nulliparous
1,156962,WT,1107,Tissue,1,Nulliparous
2,156966,PS,1266,Tissue,1,Nulliparous
3,156964,IF,1366,Tissue,1,Nulliparous
4,156965,IF,1367,Tissue,1,Nulliparous


In [103]:
test_df = df_key[df_key['AnimalID'] == 'SD211']

In [105]:
test_left_df = test_df[test_df['MapBase'] == 'SD211-Left']

In [106]:
test_left_df.head()

,Image,Genotype,AnimalID,Tissue.ID,MappingID,MapBase
982,205892,WT,SD211,Tissue1,3,SD211-Left
983,205892,WT,SD211,Tissue2,1,SD211-Left
984,205893,WT,SD211,Tissue,2,SD211-Left


In [82]:
grey_values = grey_value_df[['Mapping_ID','Grey_value']]

In [96]:
grey_values.head()

,Mapping_ID,Grey_value
0,3,195
1,4,183
2,2,172
3,1,160
4,1.2,150


In [16]:
flip_series = []
rotation_series = []
save_path_cropped_masks = []
padded_imgs = []
padded_masks = []

image_names = test_left_df['Image'].to_list()
tissue_ids = test_left_df['Tissue.ID'].to_list()
map_ids = test_left_df['MappingID'].to_list()
map_base = test_left_df['MapBase'].to_list()

# test_name = str(image_names[0])
# test_tissue_id = tissue_ids[0]
# test_map_id = map_ids[0]
# map = map_base[0]

In [ ]:
for name, tissue_id, map_region,map in zip(image_names,tissue_ids,map_ids,map_base):
    img_path = os.path.join(data_path,tissue_id,str(name)+'.svs.png')
    img = np.array(Image.open(img_path).convert('L'))
    img_bbox, cropped_mask = get_tissue_bbox(img,name,tissue_id,data_path)
    pad_img_bbox, pad_mask_bbox = add_padding(img_bbox,cropped_mask,name,data_path)
    points = get_cardinal_points(img_bbox,name)
    flip, rotation = orient_tissue(points,img_bbox)
    flip_series.append(flip)
    rotation_series.append(rotation)
    save_path_cropped_masks.append(mask_save_path)
    padded_imgs.append(pad_img_bbox)
    padded_masks.append(pad_mask_bbox)

In [ ]:
viewer = napari.view_image(pad_img_bbox)

In [ ]:
map_region = get_map_region(map_base=map,map_id=test_map_id,data_path=data_path)

In [ ]:
map_spacing = calculate_spacing(pad_mask_bbox,map_region)

In [ ]:
map_spacing

In [ ]:
fill_holes_moving = sc.ndimage.binary_fill_holes(pad_mask_bbox)

In [ ]:
sitk_moving = load_sitk_moving_img(fill_holes_moving,(1.0,1.0))
sitk_fixed = load_sitk_fixed_img(map_region,map_spacing)

In [ ]:
first_transform = apply_flip_rotation(sitk_moving,sitk_fixed,data_path,test_tissue_id,test_name,flip,rotation)

In [ ]:
best_transform = multi_start_registration(sitk_moving,sitk_fixed,data_path=None,tissue_id=None,composite_transform=first_transform)

In [ ]:
resampled = sitk.Resample(
        sitk_moving, sitk_fixed, best_transform,
        sitk.sitkNearestNeighbor, 0.0, sitk_moving.GetPixelID()
    )
fig, axes = plt.subplots()
fixed   = sitk.GetArrayFromImage(sitk_fixed)
moved   = sitk.GetArrayFromImage(resampled)
axes.imshow(fixed, cmap="gray")
axes.imshow(moved,cmap="Reds",alpha=0.5)

plt.tight_layout()
#plt.savefig("registration_verification_fullmap.png", dpi=150)
plt.show()